# Reaction sets across biomass floors

Edit INPUT_DIR and OUT. Inputs are reaction_inventory.tsv and the 36 paper-network FVA
tables produced by objective_fraction_sensitivity.ipynb. They are not bundled. This
notebook reads saved tables only; it does not optimize models. Dependencies: NumPy and
pandas.

Use original nonboundary reactions and tolerance 1e-4 model flux units. Flux-capable
means either FVA endpoint has absolute value above tolerance. Required-active means
minimum > tolerance or maximum < −tolerance. These sets describe feasible activity,
not simultaneous activity of all members. Compare all six pairs among 0%, 1%, 10% and
25%. Retained = intersection, gained = later minus earlier, lost = earlier minus
later. Jaccard = intersection size / union size, with two empty sets assigned
similarity 1. Export exact identities and per-reaction memberships, directions and
endpoints. No analysis was executed for this copy.


In [ ]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd

INPUT_DIR = Path("results/objective_sensitivity_results")
OUT = Path("results/reaction_set_comparison")
OUT.mkdir(parents=True, exist_ok=True)
STAGES = ["egg", "L1", "L2", "L3", "L4f", "L4m", "Am", "Af", "L4mf_union"]
FRACTIONS = [0.0, 0.01, 0.10, 0.25]
TOL = 1e-4
inventory = pd.read_csv(INPUT_DIR / "reaction_inventory.tsv", sep="\t")
memberships = []
comparisons = []


In [ ]:
for stage in STAGES:
    reactions = inventory[(inventory.model == stage) & (~inventory.boundary)]
    membership = reactions[["model", "reaction_id", "name", "subsystem"]].copy()
    membership = membership.set_index("reaction_id").sort_index()
    sets = {}
    for fraction in FRACTIONS:
        percent = int(fraction * 100)
        ranges = (
            pd.read_csv(
                INPUT_DIR / f"fva_{stage}_paper_network_{percent:02d}pct.tsv", sep="\t"
            )
            .set_index("reaction_id")
            .loc[membership.index]
        )
        assert np.isfinite(ranges[["minimum", "maximum"]].to_numpy()).all()
        capable = (ranges.minimum.abs() > TOL) | (ranges.maximum.abs() > TOL)
        required = (ranges.minimum > TOL) | (ranges.maximum < -TOL)
        sets["flux_capable", fraction] = set(ranges.index[capable])
        sets["required_active", fraction] = set(ranges.index[required])
        membership[f"flux_capable_{percent}pct"] = capable
        membership[f"required_active_{percent}pct"] = required
        membership[f"required_direction_{percent}pct"] = np.select(
            [ranges.minimum > TOL, ranges.maximum < -TOL],
            ["forward", "reverse"],
            default="not_required",
        )
        membership[f"minimum_{percent}pct"] = ranges.minimum
        membership[f"maximum_{percent}pct"] = ranges.maximum
    memberships.append(membership.reset_index())
    for kind in ["flux_capable", "required_active"]:
        for earlier, later in combinations(FRACTIONS, 2):
            before, after = sets[kind, earlier], sets[kind, later]
            retained, gained, lost = before & after, after - before, before - after
            union = before | after
            comparisons.append(
                dict(
                    model=stage,
                    reaction_set=kind,
                    from_fraction=earlier,
                    to_fraction=later,
                    from_n=len(before),
                    to_n=len(after),
                    intersection_n=len(retained),
                    gained_n=len(gained),
                    lost_n=len(lost),
                    union_n=len(union),
                    jaccard=len(retained) / len(union) if union else 1.0,
                    sets_identical=before == after,
                    intersection_ids=";".join(sorted(retained)),
                    gained_ids=";".join(sorted(gained)),
                    lost_ids=";".join(sorted(lost)),
                )
            )
pd.concat(memberships, ignore_index=True).to_csv(
    OUT / "reaction_set_membership.tsv", sep="\t", index=False
)
pd.DataFrame(comparisons).to_csv(
    OUT / "reaction_set_pairwise_comparisons.tsv", sep="\t", index=False
)
